# Train the CNN verifier (workstream C)

One notebook for the whole training loop: **data → composites → group split → patches → 5-fold group CV (with plots) → final model → ONNX → test split (once) → hand-labelled hard band.**
It calls the same modules as the CLI scripts (`training/*.py`, `perception/*.py`), so results match `python -m training.train_cnn`.
Background: `training/FLAME3.md` (why composites, what the bands mean), `context.md` §10.

**Where to run it:**
* **Laptop (recommended).** The CNN is tiny (32×32 patches, about 10k parameters). A full synthetic run takes minutes on a MacBook CPU/MPS, and the data already lives here. Open in VS Code or `jupyter lab`, after `pip install -e ".[dev,train]"`.
* **Colab.** Optional, and only useful with a big patch set. Upload `patches.npz` (tens of MB) to Drive and set `patch_cache`. Don't upload raw FLAME 3 frames: see section 0.

**Smoke test:** `dataset="synth"` needs no downloads. Set `QUICK = True` for a 1-minute end-to-end check.

## 0. Configuration

In [ ]:
import os

QUICK = os.environ.get("PANDAS_NB_QUICK") == "1"   # tiny end-to-end check (CI / first run)

CONFIG = dict(
    dataset="synth",                 # "synth" (no download) | "flame3" (composites on real No-Fire frames)
    # --- FLAME 3 (dataset="flame3") ---
    flame3_raw="exclude/data/flame3_raw",              # thermal TIFFs (+ RGB for the labelling tool)
    flame3_manifest="exclude/data/flame3/manifest.csv",
    gsd_m=None,                      # metres/pixel for rows without gsd_m (FLAME3.md "GSD"); required for composites
    composites_per_background=3,
    decoys=True,                     # also insert synthetic rocks / vehicles / animals
    # --- reuse instead of rebuilding (e.g. on Colab) ---
    patch_cache=None,                # path to a patches.npz built earlier; skips sections 1-3
    # --- split / CV / model (defaults = training/train_cnn.py) ---
    test_frac=0.2, folds=5, epochs=25, batch_size=256, lr=2e-3, weight_decay=1e-4, width=16, max_pos_weight=20.0,
    min_recall=0.95,                 # SEARCH operating point (recall-first)
    min_precision=0.98,              # CONFIRM operating point (precision-first, close range)
    augment=True, scale_normalize=True, weak_labels=False,
    saturation_c=550.0,              # [VERIFY] camera gain mode (context.md section 12 item 11)
    seed=0,
    tag=None,                        # RESULTS.md section name; default = dataset name
    run_dir=None,                    # default exclude/runs/nb_<tag>
    export_to=None,                  # e.g. "perception/models/fire_cnn.onnx": set ONLY when you mean to ship
    write_results=True,              # update perception/RESULTS.md + perception/results/*.json
    run_test=False,                  # touch the TEST split (once, at the very end). See section 6.
    labels_dir="exclude/labels",     # hand labels from training/label_tool.py (section 7)
    synth_flights=16, synth_frames_per_flight=40,
)
if QUICK:
    CONFIG.update(synth_flights=6, synth_frames_per_flight=12, folds=3, epochs=2, run_test=True, write_results=False,
                  composites_per_background=1)

### Environment: laptop or Colab
On Colab this clones the (public) repo, installs it and links `exclude/data` to a Drive folder. **Never put data in the repo**: it is public, GitHub rejects files over 100 MB, and the FLAME 3 licence is the authors'.

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/nathacks-pandas-wildfire-drones")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Pandas-Wildfire-Drones/nathacks-pandas-wildfire-drones.git",
                        str(REPO)], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,train]"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/pandas-data")          # put patches.npz / small datasets here
    DRIVE.mkdir(parents=True, exist_ok=True)
    (REPO / "exclude").mkdir(exist_ok=True)
    if not (REPO / "exclude" / "data").exists():
        (REPO / "exclude" / "data").symlink_to(DRIVE)
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("repo:", REPO, "| colab:", IN_COLAB)

In [ ]:
import json, shutil, datetime as dt
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import Markdown, display

from perception.datasets import load_manifest, write_manifest, assign_group_split, load_frame, load_mask, frame_label
from perception.patches import PATCH_SPEC, normalize_patches
from perception.evaluation import average_precision, binary_metrics, _pr_curve
from perception.detectors.fused import HotBypassRule
from perception.results_doc import update_section
from training.cv import make_folds, evaluate_oof
from training.model import TinyFireNet
from training.patch_data import PatchSet, build_patch_set, augment_batch
from training.train_cnn import pick_device, seed_all, predict, export_onnx, check_onnx_parity, negative_peak_stats, cv_markdown

C = SimpleNamespace(**CONFIG)
C.tag = C.tag or (C.dataset + ("_quick" if QUICK else ""))
RUN = Path(C.run_dir or f"exclude/runs/nb_{C.tag}")
RUN.mkdir(parents=True, exist_ok=True)
device = pick_device("auto")
BANDS = (("<80", -np.inf, 80.0), ("80-200", 80.0, 200.0), (">200", 200.0, np.inf))   # FLAME 3 labeling heuristic
print(f"tag={C.tag} run_dir={RUN} device={device}")

## 1. Data
* **synth**: fully synthetic night frames (`perception/synth.py`). Exact masks; also our night test set.
* **flame3**: real FLAME 3 frames → `manifest.csv` (`training/build_manifest_flame3.py`). Then **composites**: synthetic fires inserted into the real **No-Fire** frames (`training/composite.py`). FLAME 3's own Fire/No-Fire labels are a max-temperature heuristic, so we only trust the No-Fire frames as backgrounds.

FLAME 3 first time: run `python -m training.build_manifest_flame3 --root <raw> --inspect` once in a terminal and check the printed value ranges (°C, not Kelvin) and folder labels.

In [ ]:
from perception.synth import generate_dataset
from training.composite import build_composites

if C.patch_cache:
    print("patch_cache set: skipping sections 1-3")
    MANIFEST = None
elif C.dataset == "synth":
    out = Path(f"exclude/data/{'synth_quick' if QUICK else 'synth'}")
    MANIFEST = out / "manifest.csv"
    if not MANIFEST.exists():
        generate_dataset(out, C.synth_flights, C.synth_frames_per_flight, seed=C.seed)
elif C.dataset == "flame3":
    src = Path(C.flame3_manifest)
    if not src.exists():
        raise SystemExit(f"{src} missing: python -m training.build_manifest_flame3 --root {C.flame3_raw} --out {src} --group-by sequence")
    comp_dir = src.parent.parent / "flame3_composite"
    summary = build_composites(src, comp_dir, C.composites_per_background, C.gsd_m, decoys=C.decoys, seed=C.seed)
    print(json.dumps(summary, indent=1))
    MANIFEST = Path(summary["manifest"])
else:
    raise ValueError(C.dataset)

if MANIFEST:
    recs = load_manifest(MANIFEST)
    labels = np.array([frame_label(r) for r in recs])
    print(f"{len(recs)} frames, {len({r.group for r in recs})} groups, {labels.sum()} with fire, "
          f"gsd known for {sum(r.gsd_m is not None for r in recs)}")

In [ ]:
# A few frames with their fire masks (cyan): sanity-check units and labels by eye.
if MANIFEST:
    show = [r for r in recs if frame_label(r)][:3] + [r for r in recs if not frame_label(r)][:1]
    fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 4))
    for ax, r in zip(np.atleast_1d(axes), show):
        t = load_frame(r)
        im = ax.imshow(t, cmap="inferno", vmin=np.percentile(t, 1), vmax=max(np.percentile(t, 99.9), np.percentile(t, 1) + 1))
        m = load_mask(r)
        if m is not None and m.any():
            ax.contour(m, levels=[0.5], colors="cyan", linewidths=0.8)
        ax.set_title(f"{Path(r.frame_path).stem}\nmax {t.max():.0f} °C, label {frame_label(r)}", fontsize=9)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()

## 2. Group-wise train/test split
**Why "by group":** frames from one flight or burn are near-duplicates: same spot, same fire, a few seconds apart. If frame 101 is in train and frame 102 in test, the test is a memory check, not a generalization check, and the scores inflate. So whole **groups** (burn, flight, or time block for the single-burn Kaggle subset) go to one side. Composites keep their background's group, so a background and its composites never straddle the split. The same rule applies to CV folds (section 4).

In [ ]:
if MANIFEST:
    split_path = MANIFEST.with_name("manifest_split.csv")
    recs = assign_group_split(load_manifest(MANIFEST), C.test_frac, C.seed)
    write_manifest(split_path, recs)
    train_g = {r.group for r in recs if r.split == "train"}
    test_g = {r.group for r in recs if r.split == "test"}
    assert not train_g & test_g, "a group is on both sides of the split"
    for s in ("train", "test"):
        rs = [r for r in recs if r.split == s]
        print(f"{s:5s}: {len(rs):5d} frames, {len({r.group for r in rs}):3d} groups, {sum(frame_label(r) for r in rs)} with fire")

## 3. Patches
32×32 patches centred on the **same classical proposals** the drone produces (ThresholdDetector), labelled by mask overlap, plus mask-centred positives and random background negatives (`training/patch_data.py`). With `gsd_m` on every frame, each patch is a fixed 2.3 m ground window (scale-normalized). The cache `patches.npz` is the only thing Colab needs.

In [ ]:
if C.patch_cache:
    ps = PatchSet.load(C.patch_cache)
    stats = {"patches": len(ps.y), "positives": int(ps.y.sum()), "negatives": int((ps.y == 0).sum()),
             "groups": len(set(ps.groups)), "scale_normalized": ps.scale_normalized,
             "proposal_negatives_hard": int(((ps.origin == "proposal") & (ps.y == 0)).sum())}
else:
    train_recs = load_manifest(split_path, split="train")
    ps = build_patch_set(train_recs, weak_labels=C.weak_labels, seed=C.seed, scale_normalize=C.scale_normalize)
    stats = ps.stats
    ps.save(RUN / "patches.npz")
    print(f"cached {RUN / 'patches.npz'} ({(RUN / 'patches.npz').stat().st_size / 1e6:.1f} MB)")
print(json.dumps(stats, indent=1))

q = ps.patches_c.shape[1] // 4
core_peak = ps.patches_c[:, q:-q, q:-q].reshape(len(ps.y), -1).max(1)   # ~ the blob's peak
patch_band = np.array([next(n for n, lo, hi in BANDS if lo <= p < hi) for p in core_peak])

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
rng = np.random.default_rng(0)
for row, cls in enumerate((1, 0)):
    idx = np.flatnonzero(ps.y == cls)
    for ax, i in zip(axes[row], rng.choice(idx, min(8, len(idx)), replace=False)):
        ax.imshow(ps.patches_c[i], cmap="inferno"); ax.axis("off")
        ax.set_title(f"{'fire' if cls else 'not'} {core_peak[i]:.0f}°C\n{ps.origin[i]}", fontsize=7)
plt.suptitle("random training patches (°C)"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.logspace(np.log10(max(core_peak.min(), 1)), np.log10(core_peak.max() + 1), 50)
for cls, name in ((0, "not fire"), (1, "fire")):
    ax.hist(core_peak[ps.y == cls], bins=bins, alpha=0.6, label=name)
for t in (80, 200):
    ax.axvline(t, color="k", ls="--", lw=0.8)
ax.set_xscale("log"); ax.set_xlabel("patch centre peak (°C)"); ax.set_ylabel("patches"); ax.legend()
ax.set_title("where the classes overlap is what the CNN has to learn (dashed: 80 / 200 °C)")
plt.show()
for n, _, _ in BANDS:
    sel = patch_band == n
    print(f"band {n:7s}: {int((ps.y[sel] == 1).sum()):6d} fire, {int((ps.y[sel] == 0).sum()):6d} not")

## 4. 5-fold group cross-validation
Each fold trains on 4/5 of the **groups** and predicts the held-out fifth, so every patch gets an out-of-fold (OOF) score from a model that never saw its group. Thresholds come from OOF scores (`training/cv.py`):
* **SEARCH**: recall ≥ `min_recall`, then the best precision. A missed fire costs more than an extra INVESTIGATE pass.
* **CONFIRM**: precision ≥ `min_precision`, then the best recall. A "yes" here can send an extinguisher.

`train_with_history` is `training.train_cnn.train_model` plus per-epoch validation curves (same optimizer, schedule, loss and seeds).

In [ ]:
def train_with_history(x_c, y, seed, val=None, prefix=""):
    seed_all(seed)
    rng = np.random.default_rng(seed)
    model = TinyFireNet(width=C.width).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=C.lr, weight_decay=C.weight_decay)
    steps = max(1, int(np.ceil(len(y) / C.batch_size)))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=C.lr, total_steps=C.epochs * steps)
    pos = max(int(y.sum()), 1)
    pos_weight = torch.tensor([min((len(y) - pos) / pos, C.max_pos_weight)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    hist = {"train_loss": [], "val_loss": [], "val_ap": []}
    for epoch in range(C.epochs):
        model.train()
        order = rng.permutation(len(y))
        total = 0.0
        for i in range(0, len(order), C.batch_size):
            idx = order[i:i + C.batch_size]
            xb = normalize_patches(augment_batch(x_c[idx], rng) if C.augment else x_c[idx], PATCH_SPEC)
            loss = loss_fn(model(torch.from_numpy(xb).to(device)),
                           torch.from_numpy(y[idx].astype(np.float32)).to(device)[:, None])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            total += float(loss) * len(idx)
        hist["train_loss"].append(total / len(y))
        if val is not None:
            p = np.clip(predict(model, val[0], device), 1e-6, 1 - 1e-6)
            hist["val_loss"].append(float(-np.mean(val[1] * np.log(p) + (1 - val[1]) * np.log(1 - p))))
            hist["val_ap"].append(average_precision(val[1], p))
    print(f"{prefix}final train loss {hist['train_loss'][-1]:.4f}" +
          (f", val AP {hist['val_ap'][-1]:.3f}" if hist["val_ap"] else ""))
    return model, hist

folds = make_folds(ps.y, ps.groups, C.folds, C.seed)
oof = np.full(len(ps.y), np.nan, dtype=np.float32)
histories = []
for k, (tr, va) in enumerate(folds):
    model, h = train_with_history(ps.patches_c[tr], ps.y[tr], C.seed + k, (ps.patches_c[va], ps.y[va]), f"[fold {k}] ")
    oof[va] = predict(model, ps.patches_c[va], device)
    histories.append(h)
cv = evaluate_oof(ps.y, oof, folds, C.min_recall, C.min_precision)
thr_search, thr_confirm = cv["final_threshold"]["threshold"], cv["confirm_threshold"]["threshold"]
display(Markdown(cv_markdown(C.tag, stats, cv, SimpleNamespace(min_recall=C.min_recall, epochs=C.epochs, width=C.width, augment=C.augment))))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for k, h in enumerate(histories):
    ep = np.arange(1, len(h["train_loss"]) + 1)
    axes[0].plot(ep, h["train_loss"], color=f"C{k}", label=f"fold {k} train")
    axes[0].plot(ep, h["val_loss"], color=f"C{k}", ls="--")
    axes[1].plot(ep, h["val_ap"], color=f"C{k}")
axes[0].set_title("loss (solid train, dashed validation)"); axes[0].set_xlabel("epoch"); axes[0].legend(fontsize=7)
axes[1].set_title("validation average precision"); axes[1].set_xlabel("epoch")
axes[1].text(0.02, 0.02, "validation loss rising while train falls = overfitting:\nlower `epochs` or add data",
             transform=axes[1].transAxes, fontsize=7)
thr, rec, prec = _pr_curve(ps.y, oof)
axes[2].plot(rec, prec, color="k")
for t, name, col in ((thr_search, "SEARCH", "C1"), (thr_confirm, "CONFIRM", "C3")):
    m = binary_metrics(ps.y, oof >= t)
    axes[2].scatter(m["recall"], m["precision"], color=col, zorder=3, label=f"{name} thr {t:.2f}")
axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].set_title(f"OOF precision-recall (AP {average_precision(ps.y, oof):.3f})")
axes[2].legend(); plt.tight_layout(); plt.show()

In [ ]:
# Score distributions and per-band behaviour at the SEARCH threshold.
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for cls, name in ((0, "not fire"), (1, "fire")):
    axes[0].hist(oof[ps.y == cls], bins=40, range=(0, 1), alpha=0.6, label=name, log=True)
for t, name, col in ((thr_search, "SEARCH", "C1"), (thr_confirm, "CONFIRM", "C3")):
    axes[0].axvline(t, color=col, ls="--", label=name)
axes[0].set_xlabel("OOF p(fire)"); axes[0].legend(); axes[0].set_title("score distribution")

rows, names = [], []
for n, _, _ in BANDS:
    sel = patch_band == n
    pos, neg = sel & (ps.y == 1), sel & (ps.y == 0)
    recall = float((oof[pos] >= thr_search).mean()) if pos.any() else np.nan
    fpr = float((oof[neg] >= thr_search).mean()) if neg.any() else np.nan
    rows.append((recall, fpr)); names.append(f"{n}\n({pos.sum()} fire / {neg.sum()} not)")
x = np.arange(len(rows))
axes[1].bar(x - 0.2, [r[0] for r in rows], 0.4, label="recall (fire kept)")
axes[1].bar(x + 0.2, [r[1] for r in rows], 0.4, label="false-alarm rate (not-fire kept)")
axes[1].set_xticks(x, names, fontsize=8); axes[1].set_ylim(0, 1.05); axes[1].legend(fontsize=8)
axes[1].set_title("per band of patch peak (80-200 °C is the hard band)")
plt.tight_layout(); plt.show()

## 5. Final model → ONNX
Trained on all training patches with the same settings, exported to ONNX and checked against PyTorch (parity must be < 1e-3). Commit only when `export_to` is set **and** this is a real-data run: the `.onnx`, the `.json` sidecar (thresholds, patch spec, CV summary) and `perception/RESULTS.md`.

In [ ]:
final, _ = train_with_history(ps.patches_c, ps.y, C.seed + 100, prefix="[final] ")
onnx_path = RUN / "fire_cnn.onnx"
export_onnx(final, onnx_path)
parity = check_onnx_parity(final, onnx_path, ps.patches_c)
assert parity <= 1e-3, f"ONNX does not match PyTorch ({parity:.2e}); do not ship"
bypass_check = negative_peak_stats(ps, HotBypassRule(saturation_c=C.saturation_c))
meta = {
    "model_kind": "cnn", "architecture": "TinyFireNet", "width": C.width, "patch_spec": PATCH_SPEC.to_dict(),
    "input_name": "patches", "output": "logits (apply sigmoid)", "scale_normalized": ps.scale_normalized,
    "threshold": thr_search, "threshold_rule": f"recall>={C.min_recall} then max precision (pooled OOF)",
    "confirm_threshold": thr_confirm, "reject_threshold": cv["reject_threshold"]["threshold"],
    "reject_threshold_rule": "recall>=0.99 (pooled OOF): below this a close-range candidate is FALSE_POSITIVE",
    "confirm_threshold_rule": f"precision>={C.min_precision} then max recall (pooled OOF, single patch)",
    "bypass_check": bypass_check, "cv_summary": cv["summary"], "patch_stats": stats,
    "manifest": str(MANIFEST or C.patch_cache), "tag": C.tag, "epochs": C.epochs, "seed": C.seed,
    "onnx_parity_max_abs": parity, "created": dt.datetime.now().isoformat(timespec="seconds"),
    "torch": torch.__version__, "trained_with": "training/train_verifier.ipynb",
}
(RUN / "fire_cnn.json").write_text(json.dumps(meta, indent=2, default=float))
(RUN / "cv.json").write_text(json.dumps({"cv": cv, "config": CONFIG}, indent=2, default=float))
np.savez_compressed(RUN / "oof.npz", y=ps.y, oof=oof, groups=ps.groups)
print(f"parity {parity:.2e} · bypass check {bypass_check}")
if C.export_to:
    dst = Path(C.export_to); dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(onnx_path, dst); shutil.copy(RUN / "fire_cnn.json", dst.with_suffix(".json"))
    print("exported to", dst)
if C.write_results:
    update_section("perception/RESULTS.md", f"cnn_cv_{C.tag}",
                   cv_markdown(C.tag, stats, cv, SimpleNamespace(min_recall=C.min_recall, epochs=C.epochs, width=C.width, augment=C.augment), bypass_check),
                   {"cv": cv, "meta": meta})
    print("updated perception/RESULTS.md")

## 6. Test split: ONCE, at the very end
The full mixed pipeline (threshold proposals → CNN → hot-blob bypass) on the held-out test groups, at frame and object level (`perception/evaluate.py`). Choose epochs, augmentation and thresholds from CV only. If you change anything after looking at these numbers, say so in RESULTS.md. Set `run_test=True` to run.

In [ ]:
from perception.detectors.classifiers import load_classifier
from perception.evaluate import evaluate_pipeline, _markdown

if C.run_test and MANIFEST:
    test_recs = load_manifest(split_path, split="test")
    clfs = {"cnn": load_classifier(onnx_path)}
    res = evaluate_pipeline(test_recs, clfs, HotBypassRule(saturation_c=C.saturation_c))
    res.update(label=f"{C.tag} test", manifest=str(split_path), split="test", models={"cnn": clfs["cnn"].meta})
    md_txt = _markdown(res, f"{C.tag} test", clfs)
    display(Markdown(md_txt))
    if C.write_results:
        update_section("perception/RESULTS.md", f"pipeline_test_{C.tag}", md_txt, res)
else:
    print("run_test=False: test split untouched")

## 7. Hand-labelled hard band (real fires, 80-200 °C)
Labels from `python -m training.label_tool label ...`, judged from the RGB photo, so they're independent of the threshold that made FLAME 3's labels. This is the only real-fire check of the hard band. "Fire" can mean *active only* or *active + smouldering*; both are reported. `unsure` is excluded.

In [ ]:
from training.label_tool import load_candidates, load_labels, labelled_targets, predict_candidates

lab_dir = Path(C.labels_dir)
cand_csv = lab_dir / "candidates.csv"
label_files = sorted(lab_dir.glob("labels_*.csv")) if lab_dir.exists() else []
if cand_csv.exists() and label_files:
    cands = load_candidates(cand_csv)
    clf = load_classifier(onnx_path)
    for lf in label_files:
        labels = load_labels(lf)
        for positive in (("active",), ("active", "smoulder")):
            kept, y = labelled_targets(cands, labels, positive)
            if len(kept) == 0:
                continue
            p = predict_candidates(clf, kept)
            for t, name in ((thr_search, "SEARCH"), (thr_confirm, "CONFIRM")):
                m = binary_metrics(y, p >= t)
                print(f"{lf.stem:18s} fire={'+'.join(positive):16s} {name:7s} n={len(y):4d} fire={y.sum():4d} "
                      f"precision {m['precision']:.3f} recall {m['recall']:.3f} f1 {m['f1']:.3f}")
        cats = {}
        for c in cands:
            if c.cand_id in labels:
                cats.setdefault(labels[c.cand_id]["label"], []).append(c)
        fig, ax = plt.subplots(figsize=(7, 3))
        for k, (cat, cs) in enumerate(sorted(cats.items())):
            ax.scatter(predict_candidates(clf, cs), np.full(len(cs), k) + np.random.default_rng(k).uniform(-.2, .2, len(cs)), s=8)
        ax.set_yticks(range(len(cats)), sorted(cats)); ax.axvline(thr_search, color="C1", ls="--", label="SEARCH")
        ax.axvline(thr_confirm, color="C3", ls="--", label="CONFIRM"); ax.set_xlabel("p(fire)"); ax.legend(fontsize=8)
        ax.set_title(f"CNN scores by hand label ({lf.stem})"); plt.tight_layout(); plt.show()
else:
    print(f"no hand labels in {lab_dir}: see training/label_tool.py (candidates -> label -> agree)")

## 8. Save
On Colab, copy the run to Drive (the VM is wiped when the session ends). On a laptop, everything is already in `exclude/runs/` (gitignored). Commit `perception/models/fire_cnn.onnx` + `.json` and `perception/RESULTS.md` only for a real-data model you intend to ship.

In [ ]:
if IN_COLAB:
    dst = DRIVE / "runs" / RUN.name
    shutil.copytree(RUN, dst, dirs_exist_ok=True)
    print("copied run to", dst)
print("run artefacts:", sorted(p.name for p in RUN.iterdir()))